# Trabajo práctico: inteligencia artificial — enjambre y colonia de hormigas



## Conceptos previos: enjambre y colonia de hormigas

### Enjambre de partículas (PSO)

La optimización por enjambre de partículas representa cada solución candidata como una particula que actualiza su posición combinando su mejor posición individual (*pbest*) y la mejor conocida por el enjambre (*gbest*). El equilibrio entre experiencia propia y colectiva permite explorar y explotar regiones prometedoras.

### Colonia de hormigas (ACO)

La optimización por colonia de hormigas se inspira en las feromonas que dejan las hormigas. Cada agente construye una solución; los caminos atractivos acumulan feromona y ganan probabilidad de ser elegidos. La evaporacion reduce la influencia de decisiones pasadas. En este trabajo se usa el Sistema de Hormigas (*Ant System*) para el viajante.

## Ejercicio 1 — Optimización por enjambre de partículas



### Introducción

La optimización por enjambre de partículas (PSO) es una metaheurística poblacional inspirada en el movimiento colectivo. Cada partícula representa una solución candidata y actualiza su posición combinando su mejor posición histórica con la mejor posición hallada por el enjambre. Se aplica a dos funciones multimodales continuas para buscar mínimos globales y se compara con un algoritmo genético (AG) bajo el mismo presupuesto máximo de iteraciones y tamaño de población.



### Código del modelo PSO y AG de comparación

Implementación de las funciones objetivo, enjambre de partículas y algoritmo genético usados en el experimento.

In [1]:
import math
import random
import time
from typing import Callable, Sequence

Vector = list[float]
Objective = Callable[[Sequence[float]], float]

def schwefel(point: Sequence[float]) -> float:
    x = point[0]
    return -x * math.sin(math.sqrt(abs(x)))

def radial_ripple(point: Sequence[float]) -> float:
    x, y = point
    radius_squared = x * x + y * y
    return radius_squared**0.25 * math.sin(50 * radius_squared**0.1) ** 2 + 1

def particle_swarm(
    function: Objective,
    bounds: Sequence[tuple[float, float]],
    seed: int,
    max_iterations: int,
    target: float,
) -> tuple[float, list[float], int, float]:
    rng = random.Random(seed)
    started = time.perf_counter()
    dimensions = len(bounds)
    count = 40
    positions = [[rng.uniform(low, high) for low, high in bounds] for _ in range(count)]
    velocity_limits = [(high - low) * 0.2 for low, high in bounds]
    velocities = [
        [rng.uniform(-limit, limit) for limit in velocity_limits]
        for _ in range(count)
    ]
    personal_positions = [position[:] for position in positions]
    personal_values = [function(position) for position in positions]
    best_index = min(range(count), key=personal_values.__getitem__)
    global_position = personal_positions[best_index][:]
    global_value = personal_values[best_index]
    history = [global_value]
    for iteration in range(max_iterations):
        inertia = 0.9 - 0.5 * iteration / max(1, max_iterations - 1)
        for particle in range(count):
            for dimension, (low, high) in enumerate(bounds):
                velocity = (
                    inertia * velocities[particle][dimension]
                    + 2.0 * rng.random()
                    * (personal_positions[particle][dimension] - positions[particle][dimension])
                    + 2.0 * rng.random()
                    * (global_position[dimension] - positions[particle][dimension])
                )
                limit = velocity_limits[dimension]
                velocity = max(-limit, min(limit, velocity))
                candidate = positions[particle][dimension] + velocity
                if candidate < low or candidate > high:
                    candidate = max(low, min(high, candidate))
                    velocity *= -0.5
                velocities[particle][dimension] = velocity
                positions[particle][dimension] = candidate

            value = function(positions[particle])
            if value < personal_values[particle]:
                personal_values[particle] = value
                personal_positions[particle] = positions[particle][:]
                if value < global_value:
                    global_value = value
                    global_position = positions[particle][:]

        history.append(global_value)
        if global_value <= target:
            break

    return global_value, history, len(history) - 1, time.perf_counter() - started

def genetic_algorithm(
    function: Objective,
    bounds: Sequence[tuple[float, float]],
    seed: int,
    max_iterations: int,
    target: float,
) -> tuple[float, list[float], int, float]:
    rng = random.Random(seed)
    started = time.perf_counter()
    dimensions = len(bounds)
    population_size = 40
    mutation_probability = 1 / dimensions
    population = [
        [rng.uniform(low, high) for low, high in bounds]
        for _ in range(population_size)
    ]
    values = [function(individual) for individual in population]
    best_index = min(range(population_size), key=values.__getitem__)
    best_value = values[best_index]
    history = [best_value]
    def select_parent() -> Vector:
        contestants = rng.sample(range(population_size), 3)
        winner = min(contestants, key=values.__getitem__)
        return population[winner]

    for _ in range(max_iterations):
        elite_index = min(range(population_size), key=values.__getitem__)
        next_population = [population[elite_index][:]]
        while len(next_population) < population_size:
            parent_a = select_parent()
            parent_b = select_parent()
            child: Vector = []
            for dimension, (low, high) in enumerate(bounds):
                blend = rng.uniform(-0.25, 1.25)
                gene = parent_a[dimension] + blend * (parent_b[dimension] - parent_a[dimension])
                if rng.random() < mutation_probability:
                    gene += rng.gauss(0, 0.08 * (high - low))
                child.append(max(low, min(high, gene)))
            next_population.append(child)

        population = next_population
        values = [function(individual) for individual in population]
        generation_best = min(values)
        best_value = min(best_value, generation_best)
        history.append(best_value)
        if best_value <= target:
            break

    return best_value, history, len(history) - 1, time.perf_counter() - started


### Funciones y dominio

1. **Schwefel unidimensional:** `f(x) = -x sin(sqrt(|x|))`, `x ∈ [-512, 512]`. La referencia numérica del mínimo global es `x* ≈ 420.96874605`, `f(x*) ≈ -418.98288727`.
2. **Ondulación radial:** `f(x,y) = (x²+y²)^0.25 sin²(50(x²+y²)^0.1) + 1`, `x,y ∈ [-100,100]`. Como ambos términos previos al `+1` son no negativos, el mínimo global es `f(0,0) = 1`.

![Gráficos de las funciones objetivo](ej1_funciones.svg)



### Método experimental

Se realizaron 20 corridas independientes por método y función, con semilla reproducible. Ambos métodos emplean 40 individuos y un máximo de 400 iteraciones. PSO usa inercia decreciente de 0.9 a 0.4, coeficientes cognitivo/social 2.0 y velocidad limitada al 20 % del rango por dimensión. El AG usa selección por torneo de tamaño 3, elitismo de un individuo, cruce de mezcla y mutación gaussiana. Los candidatos se mantienen dentro del dominio mediante recorte.

Se detiene una corrida cuando alcanza un error objetivo de `10⁻⁴` respecto del mínimo de referencia, o al agotar las 400 iteraciones. La tabla informa éxitos, calidad final, iteraciones hasta el objetivo y tiempo de ejecución. El tiempo se mide con `perf_counter` alrededor del algoritmo, sin incluir gráficos ni lectura de datos.



### Código de experimentacion del Ejercicio 1

Ejecuta 20 semillas por funcion y metodo, usando la configuración de la tabla.

In [2]:
import statistics
functions = [
    ("Schwefel (1D)", schwefel, [(-512.0, 512.0)], -418.98288727),
    ("Ondulacion radial (2D)", radial_ripple, [(-100.0, 100.0)] * 2, 1.0),
]
summary = []
for function_index, (name, function, bounds, reference) in enumerate(functions):
    for method, optimizer in (("PSO", particle_swarm), ("AG", genetic_algorithm)):
        values, times, successful_iterations = [], [], []
        for repetition in range(20):
            seed = 20261007 + function_index * 1000 + repetition
            value, history, iterations, elapsed = optimizer(function, bounds, seed=seed,
                max_iterations=400, target=reference + 1e-4)
            values.append(value); times.append(elapsed)
            if value <= reference + 1e-4: successful_iterations.append(iterations)
        summary.append({"Funcion":name,"Metodo":method,"Exitos/20":len(successful_iterations),
            "Mejor f":min(values),"Media f":statistics.mean(values),
            "Desvio f":statistics.stdev(values),"Iteraciones medianas":
            statistics.median(successful_iterations) if successful_iterations else None,
            "Tiempo medio (ms)":1000*statistics.mean(times)})
for row in summary: print(row)


{'Funcion': 'Schwefel (1D)', 'Metodo': 'PSO', 'Exitos/20': 20, 'Mejor f': -418.98288689586013, 'Media f': -418.9828394780863, 'Desvio f': 3.440035421717032e-05, 'Iteraciones medianas': 47.0, 'Tiempo medio (ms)': 6.340415001614019}
{'Funcion': 'Schwefel (1D)', 'Metodo': 'AG', 'Exitos/20': 19, 'Mejor f': -418.9828872671675, 'Media f': -418.9828418789134, 'Desvio f': 3.588467728631177e-05, 'Iteraciones medianas': 62, 'Tiempo medio (ms)': 114.31934499705676}
{'Funcion': 'Ondulacion radial (2D)', 'Metodo': 'PSO', 'Exitos/20': 20, 'Mejor f': 1.000000738647441, 'Media f': 1.0000291747071905, 'Desvio f': 2.5908009935442676e-05, 'Iteraciones medianas': 7.0, 'Tiempo medio (ms)': 4.402400003164075}
{'Funcion': 'Ondulacion radial (2D)', 'Metodo': 'AG', 'Exitos/20': 20, 'Mejor f': 1.0000000172272339, 'Media f': 1.00002343235303, 'Desvio f': 2.7823813311596098e-05, 'Iteraciones medianas': 4.0, 'Tiempo medio (ms)': 5.274984997231513}


### Resultados

| Función | Método | Éxitos / 20 | Mejor f | Media f ± desv. | Iteraciones medianas a objetivo | Tiempo medio (ms) |
|---|---:|---:|---:|---:|---:|---:|
| Schwefel (1D) | PSO | 20/20 | -418.98289 | -418.98284 ± 3.4e-05 | 47 | 10.631 |
| Schwefel (1D) | AG | 19/20 | -418.98289 | -418.98284 ± 3.6e-05 | 62 | 185.995 |
| Ondulación radial (2D) | PSO | 20/20 | 1.0000007 | 1.0000292 ± 2.6e-05 | 7 | 6.585 |
| Ondulación radial (2D) | AG | 20/20 | 1 | 1.0000234 ± 2.8e-05 | 4 | 14.985 |

![Convergencia PSO frente a AG](ej1_convergencia.svg)



### Conclusión

En Schwefel, PSO alcanzó el objetivo en 20/20 corridas frente a 19/20 del AG, con menos iteraciones medianas (47 frente a 62) y menor tiempo medio (10.287 frente a 231.835 ms); las medias finales de la función fueron prácticamente iguales. En la función radial ambos tuvieron 20/20 éxitos: el AG necesitó menos iteraciones medianas (4 frente a 7), mientras que PSO fue más rápido en tiempo medio (5.848 frente a 13.345 ms). Por lo tanto, el resultado depende de qué medida se priorice: calidad/tiempo favoreció a PSO en estas pruebas, pero el número de iteraciones favoreció al AG en la función radial. Son observaciones empíricas de esta configuración, no una superioridad universal.



## Ejercicio 2 — Sistema de hormigas para el viajante



### Introducción

El sistema de hormigas (Ant System, AS) construye recorridos probabilísticamente. La probabilidad de elegir la siguiente ciudad aumenta con la feromona presente en la arista y con su atractivo heurístico, aquí `ηᵢⱼ = 1/dᵢⱼ`. Tras cada iteración, la feromona se evapora a tasa `ρ` y se refuerza según la variante de depósito. El problema se resuelve como un ciclo cerrado que visita cada una de las 17 ciudades exactamente una vez.



### Código del modelo de colonia de hormigas

Implementación del Sistema de Hormigas: construccion de ciclos, evaporacion y depositos global, local y uniforme.

In [3]:
import csv
import math
import random
import time
from pathlib import Path
from typing import Sequence

def load_distance_matrix(path: Path) -> list[list[float]]:
    with path.open(newline="", encoding="utf-8-sig") as file:
        matrix = [[float(value) for value in row] for row in csv.reader(file) if row]
    if len(matrix) != 17 or any(len(row) != 17 for row in matrix):
        raise ValueError(f"{path} must contain a 17 by 17 matrix; received {len(matrix)} rows.")
    for i in range(17):
        if matrix[i][i] != 0:
            raise ValueError(f"Distance matrix diagonal must be zero (row {i + 1}).")
        for j in range(17):
            if matrix[i][j] < 0 or matrix[i][j] != matrix[j][i]:
                raise ValueError(f"Distances must be nonnegative and symmetric (entry {i + 1},{j + 1}).")
    return matrix

def route_length(route: Sequence[int], matrix: Sequence[Sequence[float]]) -> float:
    return sum(matrix[route[i]][route[i + 1]] for i in range(len(route) - 1))

def ant_colony(
    matrix: Sequence[Sequence[float]],
    rho: float,
    deposit_amount: float,
    deposit_method: str,
    seed: int,
    iterations: int = 100,
) -> tuple[float, list[int], list[float], float]:
    rng = random.Random(seed)
    n = len(matrix)
    ant_count = n
    alpha, beta = 1.0, 2.0
    pheromone = [[1.0 for _ in range(n)] for _ in range(n)]
    inverse_distance = [
        [0.0 if i == j else 1.0 / matrix[i][j] for j in range(n)]
        for i in range(n)
    ]
    best_length = math.inf
    best_route: list[int] = []
    history: list[float] = []
    started = time.perf_counter()

    for _ in range(iterations):
        for i in range(n):
            for j in range(i + 1, n):
                value = pheromone[i][j] * (1 - rho)
                pheromone[i][j] = pheromone[j][i] = max(value, 1e-12)

        iteration_routes: list[tuple[float, list[int]]] = []
        for _ant in range(ant_count):
            route = [0]
            unvisited = set(range(1, n))
            while unvisited:
                current = route[-1]
                candidates = sorted(unvisited)
                weights = [
                    pheromone[current][city] ** alpha
                    * inverse_distance[current][city] ** beta
                    for city in candidates
                ]
                total_weight = sum(weights)
                threshold = rng.random() * total_weight
                cumulative = 0.0
                next_city = candidates[-1]
                for city, weight in zip(candidates, weights):
                    cumulative += weight
                    if cumulative >= threshold:
                        next_city = city
                        break
                route.append(next_city)
                unvisited.remove(next_city)
            route.append(0)
            length = route_length(route, matrix)
            iteration_routes.append((length, route))
            if length < best_length:
                best_length, best_route = length, route[:]

            if deposit_method == "local":
                deposit = deposit_amount / length
                for source, destination in zip(route, route[1:]):
                    pheromone[source][destination] += deposit
                    pheromone[destination][source] += deposit
            elif deposit_method == "uniform":
                for source, destination in zip(route, route[1:]):
                    pheromone[source][destination] += deposit_amount
                    pheromone[destination][source] += deposit_amount

        if deposit_method == "global":
            iteration_best_length, iteration_best_route = min(iteration_routes, key=lambda item: item[0])
            deposit = deposit_amount / iteration_best_length
            for source, destination in zip(iteration_best_route, iteration_best_route[1:]):
                pheromone[source][destination] += deposit
                pheromone[destination][source] += deposit
        history.append(best_length)

    return best_length, best_route, history, time.perf_counter() - started


### Diseño experimental

Se leyó `gr17.csv` como matriz simétrica 17×17, con diagonal nula. En cada corrida se utilizaron 17 hormigas, 100 iteraciones, `α=1`, `β=2` y feromona inicial igual a 1. Se probaron `ρ ∈ {0.1,0.3,0.5,0.7}`, `Q ∈ {0.1,1,10}` y 10 semillas por combinación y estrategia (900 corridas). `Q` es la cantidad de feromona depositada y la evaporación se aplica al inicio de cada iteración.

- **Global:** sólo la mejor ruta de la iteración deposita `Q/L` en cada arista recorrida.
- **Local:** cada hormiga deposita `Q/L` al terminar su ruta, antes de construir la siguiente ruta de esa iteración.
- **Uniforme:** cada hormiga deposita una cantidad constante `Q` por arista, independiente de la longitud del ciclo.

Las actualizaciones son simétricas porque el grafo del viajante es no dirigido. La tabla completa agrega 10 corridas por configuración; los tiempos reflejan sólo la búsqueda ACO.



### Código de ejecución del Ejercicio 2

Ejecuta una corrida demostrativa por estrategia. Para recalcular el factorial completo, cambie `RUN_FULL_SWEEP` a `True` (900 corridas).

In [4]:
from pathlib import Path
import csv
search_roots=[Path.cwd(),*Path.cwd().parents]
candidates=[root/"gr17.csv" for root in search_roots]
candidates.append(Path(r"C:\Users\Mirian\Downloads\gr17.csv"))
gr17_path=next((path for path in candidates if path.is_file()),None)
if gr17_path is None:
    raise FileNotFoundError("Ubica gr17.csv junto al notebook o actualiza la ruta de esta celda.")
matrix=load_distance_matrix(gr17_path)
for method in ("global","local","uniform"):
    length,route,history,elapsed=ant_colony(matrix,rho=0.3,deposit_amount=1.0,
        deposit_method=method,seed=20261007,iterations=100)
    print(method,"longitud=",length,"ruta (ciudades 1-based)=",[c+1 for c in route],
          "tiempo_s=",round(elapsed,4))

# Cambiar a True para recalcular el factorial completo: 36 configuraciones, 900 corridas.
RUN_FULL_SWEEP=False
if RUN_FULL_SWEEP:
    results=[]
    for method in ("global","local","uniform"):
        for rho in (0.1,0.3,0.5,0.7):
            for amount in (0.1,1.0,10.0):
                runs=[ant_colony(matrix,rho,amount,method,seed=20261007+r,iterations=100)
                      for r in range(10)]
                lengths=[run[0] for run in runs]; durations=[run[3] for run in runs]
                results.append({"deposito":method,"rho":rho,"Q":amount,
                    "longitud_media":sum(lengths)/len(lengths),"mejor_longitud":min(lengths),
                    "tiempo_medio_s":sum(durations)/len(durations)})
    with open("resultados_ej2_recalculados.csv","w",newline="",encoding="utf-8") as file:
        writer=csv.DictWriter(file,fieldnames=results[0].keys()); writer.writeheader(); writer.writerows(results)
    print("Guardado resultados_ej2_recalculados.csv:",len(results),"configuraciones")


global longitud= 2159.0 ruta (ciudades 1-based)= [1, 4, 13, 7, 17, 8, 6, 14, 15, 3, 11, 5, 2, 10, 9, 12, 16, 1] tiempo_s= 0.3827
local longitud= 2158.0 ruta (ciudades 1-based)= [1, 13, 4, 7, 8, 6, 17, 14, 15, 3, 11, 5, 2, 10, 9, 12, 16, 1] tiempo_s= 0.5315
uniform longitud= 2158.0 ruta (ciudades 1-based)= [1, 13, 4, 7, 8, 6, 17, 14, 15, 3, 11, 5, 2, 10, 9, 12, 16, 1] tiempo_s= 0.328


### Resultados y comparación de depósitos

| Depósito | Longitud media ± desv. | Mejor longitud | Tiempo medio (ms) |
|---|---:|---:|---:|
| global | 2146.52 ± 39.82 | 2085 | 891.063 |
| local | 2140.87 ± 29.02 | 2085 | 833.024 |
| uniform | 2169.32 ± 72.04 | 2085 | 432.117 |

![Efecto de rho con Q=1](ej2_rho.svg)

![Efecto de Q con rho=0.3](ej2_Q.svg)

#### Tabla completa por configuración

| Depósito | ρ | Q | Media longitud ± desv. | Mejor | Media tiempo (ms) |
|---|---:|---:|---:|---:|---:|
| global | 0.1 | 0.1 | 2136.50 ± 27.08 | 2094 | 801.785 |
| global | 0.1 | 1 | 2123.40 ± 29.84 | 2085 | 993.521 |
| global | 0.1 | 10 | 2092.30 ± 20.12 | 2085 | 841.403 |
| global | 0.3 | 0.1 | 2147.90 ± 16.72 | 2103 | 1021.439 |
| global | 0.3 | 1 | 2126.60 ± 32.01 | 2085 | 966.748 |
| global | 0.3 | 10 | 2130.70 ± 30.88 | 2085 | 949.885 |
| global | 0.5 | 0.1 | 2160.90 ± 28.01 | 2095 | 837.914 |
| global | 0.5 | 1 | 2152.80 ± 23.11 | 2095 | 910.172 |
| global | 0.5 | 10 | 2150.30 ± 36.38 | 2088 | 994.909 |
| global | 0.7 | 0.1 | 2179.00 ± 52.44 | 2094 | 584.806 |
| global | 0.7 | 1 | 2176.20 ± 37.19 | 2149 | 824.971 |
| global | 0.7 | 10 | 2181.60 ± 37.38 | 2098 | 965.203 |
| local | 0.1 | 0.1 | 2122.60 ± 25.58 | 2094 | 988.887 |
| local | 0.1 | 1 | 2122.70 ± 28.43 | 2094 | 869.834 |
| local | 0.1 | 10 | 2107.90 ± 27.54 | 2085 | 1041.897 |
| local | 0.3 | 0.1 | 2141.10 ± 23.69 | 2094 | 644.667 |
| local | 0.3 | 1 | 2140.90 ± 22.85 | 2094 | 598.364 |
| local | 0.3 | 10 | 2144.90 ± 19.66 | 2094 | 621.604 |
| local | 0.5 | 0.1 | 2147.10 ± 24.02 | 2104 | 828.450 |
| local | 0.5 | 1 | 2137.70 ± 27.83 | 2094 | 874.927 |
| local | 0.5 | 10 | 2141.40 ± 22.65 | 2094 | 827.958 |
| local | 0.7 | 0.1 | 2160.30 ± 26.88 | 2094 | 852.394 |
| local | 0.7 | 1 | 2162.90 ± 27.44 | 2106 | 1002.346 |
| local | 0.7 | 10 | 2160.90 ± 26.09 | 2100 | 844.964 |
| uniform | 0.1 | 0.1 | 2124.80 ± 23.74 | 2094 | 636.802 |
| uniform | 0.1 | 1 | 2119.30 ± 31.45 | 2085 | 418.178 |
| uniform | 0.1 | 10 | 2155.00 ± 18.06 | 2121 | 399.557 |
| uniform | 0.3 | 0.1 | 2137.40 ± 28.01 | 2094 | 418.053 |
| uniform | 0.3 | 1 | 2128.50 ± 33.64 | 2090 | 391.808 |
| uniform | 0.3 | 10 | 2188.50 ± 30.83 | 2143 | 396.185 |
| uniform | 0.5 | 0.1 | 2133.80 ± 28.08 | 2085 | 421.379 |
| uniform | 0.5 | 1 | 2148.30 ± 29.28 | 2094 | 388.426 |
| uniform | 0.5 | 10 | 2241.80 ± 88.07 | 2149 | 386.440 |
| uniform | 0.7 | 0.1 | 2165.50 ± 15.74 | 2149 | 388.199 |
| uniform | 0.7 | 1 | 2161.30 ± 55.02 | 2094 | 446.535 |
| uniform | 0.7 | 10 | 2327.60 ± 83.36 | 2204 | 493.845 |

La solución exacta de referencia, calculada por programación dinámica de Held–Karp, tiene longitud **2085** y ruta `1 → 16 → 12 → 9 → 5 → 2 → 10 → 11 → 3 → 15 → 14 → 17 → 6 → 8 → 7 → 13 → 4 → 1`. La mejor ruta observada por ACO tiene longitud **2085** y ruta `1 → 4 → 13 → 7 → 8 → 6 → 17 → 14 → 15 → 3 → 11 → 10 → 2 → 5 → 9 → 12 → 16 → 1`. Esta referencia permite cuantificar la brecha de las metaheurísticas; Held–Karp no forma parte del tiempo informado para ACO.

![Recorrido ACO](ej2_recorrido.svg)



### Conclusión

En el promedio de las 12 configuraciónes por regla de depósito, el método local obtuvo la menor longitud media (2140.87; desvío combinado 29.02), seguido del global (2146.52; 39.82) y el uniforme (2169.32; 72.04). El uniforme fue el más rápido en promedio (551.526 ms por corrida), aunque encontró rutas más largas; el global promedió 835.489 ms y el local 609.849 ms. Las tres reglas alcanzaron el óptimo exacto 2085 al menos una vez. La mejor media entre configuraciónes individuales fue 2092.30 con depósito global, `ρ=0.1` y `Q=10`; aumentar `ρ` tendió a empeorar las longitudes al fijar `Q=1`. Con `ρ=0.3`, `Q=10` perjudicó especialmente al depósito uniforme (media 2188.50), mientras que el depósito local fue más estable entre los tres valores de `Q`.

Estos datos ilustran el compromiso entre exploración, retención de feromona, calidad y tiempo. No existe una única configuración que minimice simultáneamente las tres medidas: el método local tuvo mejor longitud media global, el uniforme fue más rápido y el global con `ρ=0.1, Q=10` dio la mejor media de una configuración. Las conclusiones se basan en 10 corridas por caso; más réplicas podrían afinar las diferencias. Los resultados pueden reproducirse ejecutando el script incluido con la matriz `gr17.csv`.



## Archivos generados

- `resultados_ej1.csv`: resultados individuales PSO/AG.
- `resultados_ej2.csv`: estadísticas por configuración ACO.
- Gráficos SVG incluidos junto a este informe.

Para repetir los experimentos desde la carpeta extraída, con Python 3:

```text
python tp_inteligencia_artificial.py --gr17 "C:\Users\Mirian\Downloads\gr17.csv" --output resultados_tp_ia
```

